In [ ]:
# Install required packages inside the notebook
!pip install -q opencv-python-headless numpy torch

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random

def get_batches(image_dir, mask_dir, batch_size=8):
    """Loads images and masks directly from folders and yields them in batches."""
    images = os.listdir(image_dir)
    random.shuffle(images)

    for i in range(0, len(images), batch_size):
        batch_img_names = images[i:i+batch_size]
        batch_images = []
        batch_masks = []

        for img_name in batch_img_names:
            img_path = os.path.join(image_dir, img_name)
            mask_name = img_name.replace('.jpg', '.png')
            mask_path = os.path.join(mask_dir, mask_name)

            # Read and format Image
            image = cv2.imread(img_path)
            if image is None:
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (1024, 1024))
            image = image.transpose(2, 0, 1)
            image = image / 255.0

            # Read and format Mask
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue
            mask = cv2.resize(mask, (1024, 1024), interpolation=cv2.INTER_NEAREST)
            mask = mask / 255.0
            mask = np.expand_dims(mask, axis=0)

            batch_images.append(image)
            batch_masks.append(mask)

        if len(batch_images) > 0:
            img_tensor = torch.tensor(np.array(batch_images), dtype=torch.float32)
            mask_tensor = torch.tensor(np.array(batch_masks), dtype=torch.float32)
            yield img_tensor, mask_tensor

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# Double Convolutional Block to first extract features and then refine them
def double_conv(in_channels, out_channels):
    """Helper to generate standard Convolutional Blocks."""
    """Padding mode is set to 'reflect' to reduce edge artifacts in the output."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, padding_mode='reflect'),
        nn.GroupNorm(32, out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, padding_mode='reflect'),
        nn.GroupNorm(32, out_channels),
        nn.ReLU(inplace=True)
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# U-Net4 has exactly 4 downsampling (pooling) layers and 4 upsampling layers
unet4_layers = nn.ModuleDict({
    # ENCODER
    'enc1': double_conv(3, 64),
    'enc2': double_conv(64, 128),
    'enc3': double_conv(128, 256),
    'enc4': double_conv(256, 512),
    
    # BOTTLENECK
    'bottleneck': double_conv(512, 1024),
    
    # DECODER UPSAMPLING STRIDES
    'up4': nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2),
    'up3': nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2),
    'up2': nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
    'up1': nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
    
    # DECODER CONVOLUTIONS (Input channels = skip_channels + upsampled_channels)
    'dec4': double_conv(1024, 512),
    'dec3': double_conv(512, 256),
    'dec2': double_conv(256, 128),
    'dec1': double_conv(128, 64),
    
    # FINAL OUTPUT
    'final_out': nn.Conv2d(64, 1, kernel_size=1)
}).to(device)



# Residual Linear Attention(since crack detection often involves thin lines/strips)
def residual_linear_attention(tensor):
    """Residual Linear Attention Module: Focuses on thin lines/strips."""
    b, c, h, w = tensor.shape
    pool_v = F.interpolate(F.adaptive_avg_pool2d(tensor, (h, 1)), size=(h, w), mode='nearest')
    pool_h = F.interpolate(F.adaptive_avg_pool2d(tensor, (1, w)), size=(h, w), mode='nearest')
    attention = torch.sigmoid(pool_v + pool_h)
    return tensor + (tensor * attention)

# U-Net4 Forward Pass with Explicit Attention on Skip Connections
def run_unet4_forward(images, layers):
    #  ENCODER (4 Pooling Layers) 
    x1 = layers['enc1'](images)
    skip1 = residual_linear_attention(x1)  # Attention applied strictly within the skip connection
    p1 = F.max_pool2d(x1, kernel_size=2)
    
    x2 = layers['enc2'](p1)
    skip2 = residual_linear_attention(x2)
    p2 = F.max_pool2d(x2, kernel_size=2)
    
    x3 = layers['enc3'](p2)
    skip3 = residual_linear_attention(x3)
    p3 = F.max_pool2d(x3, kernel_size=2)
    
    x4 = layers['enc4'](p3)
    skip4 = residual_linear_attention(x4)
    p4 = F.max_pool2d(x4, kernel_size=2)
    
    # -- BOTTLENECK --
    bot = layers['bottleneck'](p4)
    
    # -- DECODER --
    up4 = layers['up4'](bot)
    merge4 = torch.cat([skip4, up4], dim=1)
    d4 = layers['dec4'](merge4)
    
    up3 = layers['up3'](d4)
    merge3 = torch.cat([skip3, up3], dim=1)
    d3 = layers['dec3'](merge3)
    
    up2 = layers['up2'](d3)
    merge2 = torch.cat([skip2, up2], dim=1)
    d2 = layers['dec2'](merge2)
    
    up1 = layers['up1'](d2)
    merge1 = torch.cat([skip1, up1], dim=1)
    d1 = layers['dec1'](merge1)
    
    return layers['final_out'](d1)


# Custom Evaluation Metrics
def calculate_metrics(pred_logits, true_masks, threshold=0.5):
    """Calculates Precision, Recall, F1, IoU(c), and mIoU for a batch."""
    # Convert logits to 0 or 1 based on the threshold
    pred_probs = torch.sigmoid(pred_logits)
    preds = (pred_probs > threshold).float()
    
    # Flatten the tensors to 1D arrays for easy pixel-wise comparison
    preds = preds.view(-1)
    trues = true_masks.view(-1)
    
    # Core Confusion Matrix Components
    TP = (preds * trues).sum().item()
    FP = (preds * (1 - trues)).sum().item()
    FN = ((1 - preds) * trues).sum().item()
    TN = ((1 - preds) * (1 - trues)).sum().item()
    
    # Small epsilon to prevent division by zero errors
    eps = 1e-7 
    
    # 1. Precision & Recall
    precision = TP / (TP + FP + eps)
    recall = TP / (TP + FN + eps)
    
    # 2. F1 Score (Harmonic mean of P and R)
    f1 = 2 * (precision * recall) / (precision + recall + eps)
    
    # 3. Intersection over Union for Cracks (IoU_c)
    iou_c = TP / (TP + FP + FN + eps)
    
    # 4. Mean Intersection over Union (mIoU)
    iou_bg = TN / (TN + FP + FN + eps)
    miou = (iou_c + iou_bg) / 2
    
    return precision, recall, f1, iou_c, miou

In [ ]:
import torch.optim as optim

# Configuration 
TRAIN_IMG_DIR = r'deepcrack-dataset/train_img'
TRAIN_LAB_DIR = r'deepcrack-dataset/train_lab'
VAL_IMG_DIR = r'deepcrack-dataset/test_img'
VAL_LAB_DIR = r'deepcrack-dataset/test_lab'

EPOCHS = 50
BATCH_SIZE = 1
ACCUMULATION_STEPS = 8
criterion = nn.BCEWithLogitsLoss()
# Connect the U-Net4 module dictionary parameters to the Adam optimizer
optimizer = optim.Adam(unet4_layers.parameters(), lr=0.0001)

# The Training & Validation Loop 
print("Beginning Training...")

for epoch in range(EPOCHS):
    
    # TRAINING PHASE 
    # TRAINING PHASE 
    running_loss = 0.0
    train_batches = 0
    
    # 1. Zero gradients before the inner loop starts
    optimizer.zero_grad() 
    
    # 2. Add enumerate to track step count
    for i, (images, masks) in enumerate(get_batches(TRAIN_IMG_DIR, TRAIN_LAB_DIR, BATCH_SIZE)):
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward Pass
        outputs = run_unet4_forward(images, unet4_layers)
        
        # Compute Loss & Normalize it for accumulation
        loss = criterion(outputs, masks)
        loss = loss / ACCUMULATION_STEPS 
        
        # Accumulate gradients
        loss.backward()
        
        # 3. Update weights only after ACCUMULATION_STEPS have passed
        if (i + 1) % ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        # Multiply loss back by accumulation steps for accurate printing/logging
        running_loss += (loss.item() * ACCUMULATION_STEPS) 
        train_batches += 1
        
    avg_train_loss = running_loss / train_batches
    
    # VALIDATION PHASE 
    val_loss = 0.0
    val_batches = 0
    
    # Metric accumulators
    tot_p, tot_r, tot_f1, tot_iou_c, tot_miou = 0, 0, 0, 0, 0
    
    with torch.no_grad():
        for images, masks in get_batches(VAL_IMG_DIR, VAL_LAB_DIR, BATCH_SIZE):
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = run_unet4_forward(images, unet4_layers)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            
            # Fetch Batch Metrics
            p, r, f1, iou_c, miou = calculate_metrics(outputs, masks)
            
            tot_p += p
            tot_r += r
            tot_f1 += f1
            tot_iou_c += iou_c
            tot_miou += miou
            
            val_batches += 1
            
    # Calculate Epoch Averages
    avg_val_loss = val_loss / val_batches
    avg_p = tot_p / val_batches
    avg_r = tot_r / val_batches
    avg_f1 = tot_f1 / val_batches
    avg_iou_c = tot_iou_c / val_batches
    avg_miou = tot_miou / val_batches
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"    Validation Metrics -> P: {avg_p:.4f} | R: {avg_r:.4f} | F1: {avg_f1:.4f} | IoU(c): {avg_iou_c:.4f} | mIoU: {avg_miou:.4f}")

print("Training finished!")

In [ ]:
import os
import torch

# Path to save the model
save_path = "custom_unet4_trained.pth"

# Save model weights
torch.save(unet4_layers.state_dict(), save_path)

print(f"Model successfully saved to: {save_path}")
print("File exists:", os.path.exists(save_path))